# UICE-MIRNet: evaluation after 25 completed epochs

This notebook evaluates an **epoch-25 UICE-MIRNet model trained on the sequence-disjoint Trash-ICRA19 split**. All executable cells perform setup, validation, inference, visualization, or metric calculation; there are no training or optimizer calls.

The main comparison has exactly three images:

1. **Original** - the unmodified test image decoded as RGB.
2. **MIRNet** - the same image enhanced by the epoch-25 network.
3. **Final** - the MIRNet result followed by gray-world white balance, gamma correction, and CLAHE, in that order.

Run code cells 1 through 4 with the `UICE-CV CUDA (.venv)` kernel to load the epoch-25 model and stop. The fourth code cell is explicitly named **LOAD CHECKPOINT ONLY**. The notebook rejects frame-wise sequence leakage and a checkpoint trained on a different paired-data root. Comparisons are disabled by default; set `RUN_COMPARISONS=True` in section 2 only when you want to run sections 5 and 6. Re-running evaluates the same checkpoint and overwrites the selected-image evaluation files; it does not change model weights.

The architecture is implemented locally from [Sarkar et al., UICE-MIRNet guided image enhancement for underwater object detection (2024)](https://www.nature.com/articles/s41598-024-73243-9). The three classical operations here are this project's fixed post-processing experiment. This notebook evaluates enhancement only; it does not run the paper's object detector or establish detection accuracy.

**Training context:** training uses paired classical pseudo-targets on a sequence-disjoint train split, with sequence-disjoint validation used during training. Those targets are algorithm-generated, not clean underwater photographs. This notebook is valid only after retraining on the new split; the old `trash_icra` epoch-25 checkpoint was trained on a leaked frame-wise split and is retained only as a legacy artifact.


## 1. Load the evaluation code and identify the hardware

This block locates the project and imports the existing image-comparison and metric modules. Reloading the modules makes notebook reruns use the current implementations. PyTorch uses CUDA for network inference when available, otherwise the CPU; OpenCV performs the classical algorithms and metric calculations on the CPU. No training DataLoader, batch augmentation, or gradient updates are created.


In [ ]:
from pathlib import Path
from argparse import Namespace
import importlib
import json
import sys
import hashlib

import torch
import matplotlib.pyplot as plt
from PIL import Image as PILImage
from IPython.display import display, Image as DisplayImage, Markdown

project_candidates = (Path.cwd(), Path.cwd() / "UICE-CV", Path(r"C:\PYTHON 2\UICE-CV"))
PROJECT_ROOT = next((path.resolve() for path in project_candidates
                     if (path / "files (6)" / "compare_test_image.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Cannot locate the UICE-CV project and its evaluation modules.")
CODE_ROOT = PROJECT_ROOT / "files (6)"
if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))
import quality_metrics
import compare_test_image
import trash_icra_experiment
quality_metrics = importlib.reload(quality_metrics)
compare_test_image = importlib.reload(compare_test_image)
trash_icra_experiment = importlib.reload(trash_icra_experiment)

print(f"Project: {PROJECT_ROOT}")
print(f"Python: {sys.executable}")
print(f"PyTorch: {torch.__version__}")
print(f"Inference device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")


## 2. Choose the checkpoint, test image, and classical parameters

The checkpoint is fixed to `uice_mirnet_epoch25.pth`, which contains the weights after **25 completed epochs**. The best-validation checkpoint is not substituted, because this experiment specifically examines the epoch-25 result. `TEST_IMAGES=None` selects the first held-out test filename alphabetically; enter a list of test filenames to compare other images with the same settings.

The three classical algorithms operate cumulatively on the MIRNet output:

| Operation | What this implementation does | Fixed setting |
|---|---|---|
| Gray-world white balance | Scales each color channel so its mean approaches the average of the three channel means; clips to the 8-bit range. This assumes the scene's average color should be neutral. | Gains computed from the MIRNet image |
| Gamma correction | Applies the lookup-table mapping `output = 255 * (input / 255) ** (1 / gamma)`. With this convention, gamma above 1 brightens intermediate intensities. | `gamma=1.2` |
| CLAHE | Converts to Lab, applies contrast-limited local histogram equalization to lightness, then converts back to RGB. | `clipLimit=2.0`, `tileGridSize=(8, 8)` |

CLAHE uses a grid of **8 by 8 tiles**, not tiles measuring 8 pixels. OpenCV describes its local equalization and clipping in the [histogram equalization documentation](https://docs.opencv.org/4.x/d5/daf/tutorial_py_histogram_equalization.html). The parameters are held fixed rather than tuned to maximize the selected test image's score.


In [ ]:
RUN_COMPARISONS = False  # Keep inference opt-in.
# These are the new, leakage-free artifacts created by split_by_sequence.py.
DATASET_ROOT = PROJECT_ROOT / "trash_icra_sequence_split"
PAIRED_DATA_ROOT = PROJECT_ROOT / "trash_icra_paired_sequence"
CHECKPOINT = PROJECT_ROOT / "checkpoints" / "trash_icra_sequence" / "uice_mirnet_epoch25.pth"
EXPECTED_COMPLETED_EPOCHS = 25
RESULTS_DIR = PROJECT_ROOT / "results" / "epoch25_sequence_test_comparison"
TEST_IMAGES = None  # Example: ["bio0000_frame0000016.jpg"]
NUM_TEST_IMAGES = 1
POST_GAMMA = 1.2
CLAHE_CLIP_LIMIT = 2.0
CLAHE_TILE_GRID = (8, 8)

if not DATASET_ROOT.is_dir():
    raise FileNotFoundError(
        f"Missing sequence-disjoint dataset: {DATASET_ROOT}. Run split_by_sequence.py first."
    )
if not PAIRED_DATA_ROOT.is_dir():
    raise FileNotFoundError(
        f"Missing sequence-disjoint paired data: {PAIRED_DATA_ROOT}. Generate pseudo-targets after splitting."
    )
TEST_DIR = (DATASET_ROOT / "test").resolve()
print(f"Checkpoint: {CHECKPOINT}")
print(f"Test directory: {TEST_DIR}")
print(f"Evaluation outputs: {RESULTS_DIR}")


## 3. Validate the completed checkpoint and held-out image selection

This block reads checkpoint metadata on the CPU and verifies that the saved zero-based epoch index corresponds to 25 completed epochs. It records a SHA-256 checksum so the results can be traced to the exact checkpoint file. A saved training budget of 100, if present in metadata, describes the original schedule; it does not change the 25 epochs actually completed or request more training.

It then checks the selected test files and prints their original dimensions. Explicit filenames must resolve inside the test directory. Training pairs and validation images are not required for this evaluation, and no pseudo-targets are generated. Missing files or an unexpected epoch stop execution instead of silently selecting another run.


In [ ]:
if not CHECKPOINT.is_file():
    raise FileNotFoundError(f"Missing checkpoint: {CHECKPOINT}")
checkpoint_info = torch.load(CHECKPOINT, map_location="cpu", weights_only=True)
completed_epochs = int(checkpoint_info["epoch"]) + 1
if completed_epochs != EXPECTED_COMPLETED_EPOCHS:
    raise ValueError(f"Expected 25 completed epochs, found {completed_epochs}.")
if "model_state" not in checkpoint_info:
    raise ValueError("Checkpoint does not contain model weights.")
saved_data_root = checkpoint_info.get("args", {}).get("data_root")
if not saved_data_root or Path(saved_data_root).resolve() != PAIRED_DATA_ROOT.resolve():
    raise ValueError(
        f"Checkpoint was not trained on the configured sequence-disjoint paired root: {PAIRED_DATA_ROOT}"
    )
sequence_counts = trash_icra_experiment.validate_sequence_isolation(DATASET_ROOT)
print(f"Verified sequence isolation: {sequence_counts}")
checkpoint_sha256 = hashlib.sha256(CHECKPOINT.read_bytes()).hexdigest()
print(f"Verified completed epochs: {completed_epochs}")
print(f"Checkpoint SHA-256: {checkpoint_sha256}")
print(f"Saved model configuration: channels={checkpoint_info.get('args', {}).get('channels', 32)}, "
      f"num_rrb={checkpoint_info.get('args', {}).get('num_rrb', 3)}")
del checkpoint_info

if TEST_IMAGES:
    for filename in TEST_IMAGES:
        if not (TEST_DIR / filename).resolve().is_relative_to(TEST_DIR):
            raise ValueError(f"Selection must be inside the held-out test directory: {filename}")
selected_test_images = compare_test_image.select_test_images(TEST_DIR, TEST_IMAGES, NUM_TEST_IMAGES)
for path in selected_test_images:
    with PILImage.open(path) as image:
        print(f"Selected test image: {path.name}; size: {image.width} x {image.height}")


## 4. LOAD CHECKPOINT ONLY - epoch 25

This is the fourth executable code cell. It loads `uice_mirnet_epoch25.pth` on the CPU, checks that it contains 25 completed epochs, constructs the network, restores the weights, and sets evaluation mode. It stops after loading. No image is processed and no training is started. The model remains available as `epoch25_model`.


In [ ]:
# FOURTH CODE CELL: LOAD EPOCH-25 CHECKPOINT ONLY, THEN STOP.
from pathlib import Path
import sys
import torch

LOAD_PROJECT_ROOT = PROJECT_ROOT
load_code_root = str(CODE_ROOT)
if load_code_root not in sys.path:
    sys.path.insert(0, load_code_root)
from model import UICEMIRNet

LOAD_CHECKPOINT = CHECKPOINT
if not LOAD_CHECKPOINT.is_file():
    raise FileNotFoundError(f"Missing epoch-25 checkpoint: {LOAD_CHECKPOINT}")
loaded_checkpoint = torch.load(LOAD_CHECKPOINT, map_location="cpu", weights_only=True)
if int(loaded_checkpoint["epoch"]) + 1 != 25:
    raise ValueError("The checkpoint must contain exactly 25 completed epochs.")
loaded_args = loaded_checkpoint.get("args", {})
epoch25_model = UICEMIRNet(in_channels=3, channels=loaded_args.get("channels", 32),
                          num_rrb=loaded_args.get("num_rrb", 3))
epoch25_model.load_state_dict(loaded_checkpoint["model_state"])
epoch25_model.eval()
epoch25_model.requires_grad_(False)
del loaded_checkpoint
print(f"Loaded epoch-25 checkpoint: {LOAD_CHECKPOINT}")
print("Stopped after loading. No training or image inference was started.")


## 5. Run MIRNet and the classical chain; display the three-way comparison

This block runs only when `RUN_COMPARISONS=True`. With the default `False`, it prints a skipped message and performs no inference.

When enabled, the next block loads the model architecture and weights from the validated checkpoint and runs it in evaluation mode with gradient computation disabled. It processes the original resolution; temporary edge padding makes dimensions divisible by eight and is removed afterward. Predictions are clipped to `[0, 1]` and converted to 8-bit RGB images. CUDA uses BF16 when supported, otherwise FP16; CPU inference uses FP32.

The MIRNet output is passed through **white balance -> gamma -> CLAHE**. The comparison module saves the original, MIRNet output, and each cumulative classical stage as full-resolution PNGs. It also computes metrics on those image pixels. The figure below displays only the three requested endpoints; the five-stage image is retained on disk for inspecting intermediate changes.

The saved `three_way_comparison.png` is a presentation figure. Metric calculations use the full-resolution stage images, not the resized panels or their labels. The report records the checkpoint path, checksum, completed epochs, test-image paths, and classical parameters.


In [ ]:
if globals().get("RUN_COMPARISONS", False):
    epoch25_report = compare_test_image.compare(Namespace(
        checkpoint=str(CHECKPOINT),
        test_dir=str(TEST_DIR),
        images=[str(path.relative_to(TEST_DIR)) for path in selected_test_images],
        num_images=len(selected_test_images),
        output_dir=str(RESULTS_DIR),
        gamma=POST_GAMMA,
        clahe_clip_limit=CLAHE_CLIP_LIMIT,
        clahe_tile_grid=CLAHE_TILE_GRID,
    ))
    if hashlib.sha256(CHECKPOINT.read_bytes()).hexdigest() != checkpoint_sha256:
        raise RuntimeError("Checkpoint changed during evaluation; rerun with a fixed checkpoint file.")
    epoch25_report["completed_epochs"] = completed_epochs
    epoch25_report["checkpoint_sha256"] = checkpoint_sha256

    selected_stages = ("Original", "UICE-MIRNet", "+ CLAHE (final)")
    panel_titles = ("Original test image", "UICE-MIRNet - epoch 25", "MIRNet + white balance + gamma + CLAHE")
    for image_report in epoch25_report["images"]:
        fig, axes = plt.subplots(1, 3, figsize=(18, 6))
        for ax, stage, title in zip(axes, selected_stages, panel_titles):
            with PILImage.open(image_report["stage_images"][stage]) as image:
                ax.imshow(image.convert("RGB"))
            ax.set_title(title, fontsize=11)
            ax.axis("off")
        fig.suptitle(Path(image_report["test_image"]).name)
        fig.tight_layout()
        comparison_path = Path(image_report["comparison_image"]).with_name("three_way_comparison.png")
        fig.savefig(comparison_path, dpi=150, bbox_inches="tight")
        plt.close(fig)
        image_report["three_way_comparison"] = str(comparison_path)
        display(DisplayImage(filename=str(comparison_path)))
        print(f"Three-way comparison: {comparison_path}")

    metrics_path = RESULTS_DIR / "test_comparison_metrics.json"
    quality_metrics.export_metrics(epoch25_report, metrics_path)
    print(f"Full metric report: {metrics_path}")
    print(f"CSV: {metrics_path.with_suffix('.csv')}")

else:
    print("Comparisons are disabled. Set RUN_COMPARISONS=True in section 2 to run them manually.")


## 6. Compare the image-quality metrics

This block reads the saved report and displays the scores for **original, MIRNet, and final** images, followed by the change caused by the full classical chain relative to MIRNet. It can be rerun after sections 1-2 without repeating inference. The CSV/JSON retain all five stages and UIQM's components, including changes from the original and MIRNet baselines.

| Metric | Meaning in this implementation | Interpretation |
|---|---|---|
| UIQM | Weighted combination of colorfulness (UICM), sharpness (UISM), and contrast (UIConM), using the documented 10-pixel block variant. | A no-reference quality proxy; larger is conventionally preferred. |
| UCIQE | Weighted Lab chroma variation, lightness contrast, and saturation using the pipeline's existing normalized-Lab convention. | A no-reference quality proxy; compare only with the same implementation and scale. |
| Entropy | Shannon entropy of the 256-bin grayscale histogram, in bits (0-8). | Describes intensity diversity; noise can also increase it. |
| RMS contrast | Standard deviation of grayscale intensities scaled to `[0, 1]`. | Describes global contrast, not correctness or visual quality by itself. |
| Average gradient | Mean magnitude of forward intensity differences, with grayscale in `[0, 1]`. | Describes local variation; edges and noise both contribute. |
| Laplacian variance | Variance of a 3x3 Laplacian response on grayscale in `[0, 1]`. | Describes fine detail/edge response and is sensitive to noise and sharpening. |

The underwater metric references are [Panetta, Gao and Agaian, UIQM](https://www.karenpanetta.com/publications/journal-articles/human-visual-systems-inspired-underwater-image-quality-measures) and [Yang and Sowmya, UCIQE](https://doi.org/10.1109/TIP.2015.2491020). Exact numerical choices are included under `metric_definitions` in the JSON; UIQM/UCIQE implementations with different scaling or block conventions are not directly interchangeable.

There is no clean reference image for this test sample, so PSNR/SSIM are not reported. The original underwater image is the baseline, not ground truth. A higher metric does not prove improved object detection. Results for one selected test image are an example, not a dataset-wide average.


In [ ]:
if globals().get("RUN_COMPARISONS", False):
    metrics_path = RESULTS_DIR / "test_comparison_metrics.json"
    if not metrics_path.is_file():
        raise FileNotFoundError("Run section 5 to generate the epoch-25 image report first.")
    quality_report = json.loads(metrics_path.read_text(encoding="utf-8"))
    if quality_report.get("completed_epochs") != 25 or Path(quality_report["checkpoint"]).resolve() != CHECKPOINT.resolve():
        raise ValueError("This report is not the expected epoch-25 evaluation. Rerun section 5.")

    metric_columns = (("uiqm", "UIQM"), ("uciqe", "UCIQE"), ("entropy", "Entropy (bits)"),
                      ("rms_contrast", "RMS contrast"), ("average_gradient", "Average gradient"),
                      ("laplacian_variance", "Laplacian variance"))
    for image_report in quality_report["images"]:
        display(Markdown(f"**Test image:** `{Path(image_report['test_image']).name}`"))
        rows = ["| Stage | " + " | ".join(label for _, label in metric_columns) + " |",
                "|---|" + "---:|" * len(metric_columns)]
        for stage, label in (("Original", "Original"), ("UICE-MIRNet", "MIRNet (25 epochs)"),
                             ("+ CLAHE (final)", "MIRNet + three classical algorithms")):
            result = next(row for row in image_report["results"] if row["stage"] == stage)
            rows.append("| " + label + " | " + " | ".join(f"{result[key]:.5f}" for key, _ in metric_columns) + " |")
        final_result = next(row for row in image_report["results"] if row["stage"] == "+ CLAHE (final)")
        rows.append("| Change: final minus MIRNet | " + " | ".join(
            f"{final_result[key + '_change_from_mirnet']:+.5f}" for key, _ in metric_columns) + " |")
        display(Markdown(chr(10).join(rows)))
    print(f"Checkpoint: {quality_report['checkpoint']}")
    print(f"Detailed metrics and definitions: {metrics_path}")
    print(f"Spreadsheet-ready CSV: {metrics_path.with_suffix('.csv')}")

else:
    print("Comparisons are disabled. Set RUN_COMPARISONS=True in section 2 to run them manually.")


## Reading and sharing the results

The output folder for each selected test image contains `00_original.png`, `01_uice_mirnet.png`, `02_colour_corrected.png`, `03_gamma_corrected.png`, and `04_clahe_final.png`. Use `three_way_comparison.png` for the requested presentation, and `side_by_side_comparison.jpg` to inspect all intermediate stages. The shared JSON/CSV reports sit in `results/epoch25_test_comparison/`.

Inspect the three endpoints for color cast, visibility, clipping, halos, and amplified texture alongside the metric table. A final result can score higher while looking overprocessed. This evaluation reports what the fixed epoch-25 model and the specified classical chain produce; it does not establish equivalence to the published training protocol or detector results.

To evaluate another held-out image, change `TEST_IMAGES` in section 2, then rerun sections 3, 5, and 6. The report covers the images selected in the most recent run; older image folders may remain on disk. No further training is needed.
